# Silver HealthKit — Post-Deploy Constraints

Run this notebook **after** the Silver HealthKit SDP pipeline has created
its tables (first successful update). Applies:

1. **Column comments** — human-readable descriptions on every column
2. **Primary key RELY constraints** — `record_id` on each silver table
3. **Foreign key RELY constraints** — relationships between silver tables and to bronze

All constraints are informational (RELY / NOT ENFORCED) — they inform the
query optimizer and document data relationships without runtime overhead.

**Usage:** Run manually after first pipeline deploy, or schedule as a
post-deploy job task.

In [0]:
dbutils.widgets.text("catalog", "hls_fde_dev", "Catalog")
dbutils.widgets.text("schema", "dev_matthew_giglia_wearables", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
fqn = f"{catalog}.{schema}"
print(f"Applying constraints to: {fqn}")

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_health_samples
  ALTER COLUMN record_id COMMENT 'Bronze record GUID — 1:1 with source ingestion row',
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims (joins to auth.users)',
  ALTER COLUMN source_platform COMMENT 'Data source platform (apple_healthkit, synthetic, etc.)',
  ALTER COLUMN uuid COMMENT 'HealthKit sample UUID from the source device',
  ALTER COLUMN sample_type COMMENT 'HKQuantityTypeIdentifier (e.g. HeartRate, StepCount, OxygenSaturation)',
  ALTER COLUMN value COMMENT 'Measured numeric value in the units specified by the unit column',
  ALTER COLUMN unit COMMENT 'Unit of measurement (count/min, count, kcal, %, ms, m, etc.)',
  ALTER COLUMN start_ts COMMENT 'Sample measurement start timestamp (UTC)',
  ALTER COLUMN end_ts COMMENT 'Sample measurement end timestamp (UTC)',
  ALTER COLUMN source_name COMMENT 'Device or app that produced the sample (e.g. Apple Watch)',
  ALTER COLUMN source_bundle_id COMMENT 'Bundle identifier of the source app',
  ALTER COLUMN metadata COMMENT 'Optional HealthKit metadata dictionary (VARIANT, shredded)',
  ALTER COLUMN sample_date COMMENT 'Date extracted from start_ts for time-series clustering',
  ALTER COLUMN sample_hour COMMENT 'Hour-of-day (0–23) for intraday pattern analysis'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_workouts
  ALTER COLUMN record_id COMMENT 'Bronze record GUID — 1:1 with source ingestion row',
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN uuid COMMENT 'HealthKit workout UUID from the source device',
  ALTER COLUMN activity_type COMMENT 'Human-readable activity type (walking, cycling, running, etc.)',
  ALTER COLUMN activity_type_raw COMMENT 'Apple HKWorkoutActivityType raw integer enum value',
  ALTER COLUMN duration_seconds COMMENT 'Total workout duration in seconds',
  ALTER COLUMN start_ts COMMENT 'Workout start timestamp (UTC)',
  ALTER COLUMN end_ts COMMENT 'Workout end timestamp (UTC)',
  ALTER COLUMN total_distance_meters COMMENT 'Total distance covered in meters (NULL if not applicable)',
  ALTER COLUMN total_energy_burned_kcal COMMENT 'Total active energy burned in kilocalories',
  ALTER COLUMN source_name COMMENT 'Device that recorded the workout',
  ALTER COLUMN metadata COMMENT 'Optional workout metadata dictionary (VARIANT)',
  ALTER COLUMN workout_date COMMENT 'Date extracted from start_ts for clustering',
  ALTER COLUMN duration_minutes COMMENT 'Derived: duration_seconds / 60',
  ALTER COLUMN distance_km COMMENT 'Derived: total_distance_meters / 1000'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_sleep_sessions
  ALTER COLUMN record_id COMMENT 'Bronze record GUID — 1:1 with source ingestion row',
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN start_ts COMMENT 'Sleep session start timestamp (UTC, typically evening)',
  ALTER COLUMN end_ts COMMENT 'Sleep session end timestamp (UTC, typically morning)',
  ALTER COLUMN total_duration_minutes COMMENT 'Total time in bed from start to end (minutes)',
  ALTER COLUMN deep_sleep_minutes COMMENT 'Minutes spent in deep (slow-wave) sleep stage',
  ALTER COLUMN rem_sleep_minutes COMMENT 'Minutes spent in REM sleep stage',
  ALTER COLUMN core_sleep_minutes COMMENT 'Minutes spent in core (light) sleep stage',
  ALTER COLUMN awake_minutes COMMENT 'Minutes spent awake during the sleep session',
  ALTER COLUMN stages COMMENT 'Full sleep stages array (VARIANT) for granular analysis',
  ALTER COLUMN sleep_date COMMENT 'Date of sleep onset for clustering and daily aggregation',
  ALTER COLUMN num_stages COMMENT 'Number of distinct sleep stage transitions'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_activity_summaries
  ALTER COLUMN record_id COMMENT 'Bronze record GUID — 1:1 with source ingestion row',
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID from JWT claims',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN activity_date COMMENT 'Calendar date for the activity summary (one per user per day)',
  ALTER COLUMN energy_burned_kcal COMMENT 'Active energy burned (Move ring value) in kilocalories',
  ALTER COLUMN energy_goal_kcal COMMENT 'Daily Move ring goal in kilocalories',
  ALTER COLUMN exercise_minutes COMMENT 'Exercise minutes logged (Exercise ring value)',
  ALTER COLUMN exercise_goal_minutes COMMENT 'Daily Exercise ring goal in minutes',
  ALTER COLUMN stand_hours COMMENT 'Stand hours completed (Stand ring value)',
  ALTER COLUMN stand_goal_hours COMMENT 'Daily Stand ring goal in hours',
  ALTER COLUMN energy_goal_pct COMMENT 'Derived: energy_burned / energy_goal (1.0 = ring closed)',
  ALTER COLUMN exercise_goal_pct COMMENT 'Derived: exercise_minutes / exercise_goal (1.0 = ring closed)',
  ALTER COLUMN stand_goal_pct COMMENT 'Derived: stand_hours / stand_goal (1.0 = ring closed)'

In [0]:
%sql
ALTER TABLE ${catalog}.${schema}.silver_deletes
  ALTER COLUMN record_id COMMENT 'Bronze record GUID — 1:1 with source ingestion row',
  ALTER COLUMN ingested_at COMMENT 'Server-side ingestion timestamp from ZeroBus',
  ALTER COLUMN user_id COMMENT 'Authenticated user ID of the user who deleted the sample',
  ALTER COLUMN source_platform COMMENT 'Data source platform identifier',
  ALTER COLUMN deleted_uuid COMMENT 'UUID of the deleted HealthKit sample (FK to silver_health_samples.uuid)',
  ALTER COLUMN sample_type COMMENT 'HKQuantityTypeIdentifier of the deleted sample for targeted lookups'

In [0]:
%sql
-- Primary Key constraints: record_id is unique per silver row (1:1 with bronze)

ALTER TABLE ${catalog}.${schema}.silver_health_samples
  ADD CONSTRAINT silver_health_samples_pk PRIMARY KEY (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_workouts
  ADD CONSTRAINT silver_workouts_pk PRIMARY KEY (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_sleep_sessions
  ADD CONSTRAINT silver_sleep_sessions_pk PRIMARY KEY (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_activity_summaries
  ADD CONSTRAINT silver_activity_summaries_pk PRIMARY KEY (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_deletes
  ADD CONSTRAINT silver_deletes_pk PRIMARY KEY (record_id) RELY;

In [0]:
%sql
-- Foreign Keys: link silver tables back to bronze and cross-reference deletes

-- All silver tables reference the bronze source row
ALTER TABLE ${catalog}.${schema}.silver_health_samples
  ADD CONSTRAINT silver_health_samples_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_workouts
  ADD CONSTRAINT silver_workouts_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_sleep_sessions
  ADD CONSTRAINT silver_sleep_sessions_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_activity_summaries
  ADD CONSTRAINT silver_activity_summaries_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

ALTER TABLE ${catalog}.${schema}.silver_deletes
  ADD CONSTRAINT silver_deletes_bronze_fk
  FOREIGN KEY (record_id) REFERENCES ${catalog}.${schema}.wearables_zerobus (record_id) RELY;

In [0]:
print("\n" + "="*60)
print("Silver HealthKit constraints applied successfully!")
print("="*60)
print(f"\nCatalog.Schema: {fqn}")
print(f"\nTables updated:")
for t in ["silver_health_samples", "silver_workouts", "silver_sleep_sessions",
          "silver_activity_summaries", "silver_deletes"]:
    print(f"  - {fqn}.{t}")
print(f"\nApplied:")
print(f"  - Column comments on ALL columns")
print(f"  - PRIMARY KEY (record_id) RELY on all tables")
print(f"  - FOREIGN KEY -> wearables_zerobus (record_id) RELY on all tables")
print(f"\nNext: Deploy the pipeline, run first update, then run this notebook.")